# Zero-shot detector from class descriptions
Pretrained NLI weights; no prompt-injection training. Run after the comparison notebook.

In [1]:
import os

os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["USE_TF"] = "0"

import platform
import random
import re
import time
import unicodedata
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from huggingface_hub import hf_hub_download
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    average_precision_score,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from transformers import AutoModel, AutoTokenizer

OUTPUT_DIR = Path.cwd()
RANDOM_SEED = 15
MODEL_NAME = "microsoft/deberta-v3-base"
MODEL_REVISION = "8ccc9b6f36199bec6961081d44eb72fb3f7353f3"
MAX_TOKENS = 128
BATCH_SIZE = 8
VALIDATION_SHARE = 0.20
VALIDATION_FPR_LIMIT = 0.05
NEAR_DUPLICATE_SIMILARITY = 0.90
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.set_num_threads(min(4, os.cpu_count() or 1))

DATASET_SOURCES = {
    "deepset": {
        "repo": "deepset/prompt-injections",
        "revision": "4f61ecb038e9c3fb77e21034b22511b523772cdd",
        "files": {
            "train": "data/train-00000-of-00001-9564e8b05b4757ab.parquet",
            "test": "data/test-00000-of-00001-701d16158af87368.parquet",
        },
    },
    "safeguard": {
        "repo": "xTRam1/safe-guard-prompt-injection",
        "revision": "a3a877d608f37b7d20d9945671902df895ecdb46",
        "files": {
            "train": "data/train-00000-of-00001.parquet",
            "test": "data/test-00000-of-00001.parquet",
        },
    },
}
SAFEGUARD_SAMPLE_SIZES = {"train": 800, "test": 200}

hardware = platform.machine() + f" CPU; {os.cpu_count()} logical cores"
if DEVICE == "cuda":
    hardware = torch.cuda.get_device_name(0)
print("Python:", platform.python_version())
print("Hardware:", hardware)

Python: 3.13.16
Hardware: Tesla T4


In [2]:
dataset_frames = []

for dataset_name, source in DATASET_SOURCES.items():
    for publisher_split, filename in source["files"].items():
        download_path = hf_hub_download(
            source["repo"],
            filename,
            repo_type="dataset",
            revision=source["revision"], token=False,
        )
        frame = pd.read_parquet(download_path)
        frame["source_index"] = np.arange(len(frame))

        if dataset_name == "safeguard":
            sample_indices, _ = train_test_split(
                np.arange(len(frame)),
                train_size=SAFEGUARD_SAMPLE_SIZES[publisher_split],
                stratify=frame["label"],
                random_state=RANDOM_SEED,
            )
            frame = frame.iloc[np.sort(sample_indices)].copy()

        frame["dataset"] = dataset_name
        frame["publisher_split"] = publisher_split
        frame["row_id"] = [
            f"{dataset_name}:{publisher_split}:{index}"
            for index in frame["source_index"]
        ]
        dataset_frames.append(frame)

loaded_prompts = pd.concat(dataset_frames, ignore_index=True)
print("Loaded rows:", len(loaded_prompts))


def normalize_text(text):
    normalized = unicodedata.normalize("NFKC", str(text)).casefold()
    return re.sub(r"\s+", " ", normalized).strip()


loaded_prompts["normalized"] = loaded_prompts["text"].map(normalize_text)
assert loaded_prompts["label"].isin([0, 1]).all()
assert loaded_prompts["text"].notna().all()
assert loaded_prompts["normalized"].ne("").all()

audit_records = []
for dataset_name, dataset in loaded_prompts.groupby("dataset"):
    audit_records.append(
        {
            "dataset": dataset_name,
            "rows": len(dataset),
            "source_columns": 2,
            "negative": int(dataset["label"].eq(0).sum()),
            "positive": int(dataset["label"].eq(1).sum()),
            "positive_pct": 100 * dataset["label"].eq(1).mean(),
            "missing_target": int(dataset["label"].isna().sum()),
            "missing_text": int(dataset["text"].isna().sum()),
            "blank_text": int(dataset["normalized"].eq("").sum()),
            "exact_excess_text": int(dataset.duplicated("text").sum()),
            "exact_excess_text_label": int(dataset.duplicated(["text", "label"]).sum()),
            "normalized_excess": int(dataset.duplicated("normalized").sum()),
        }
    )

audit_results = pd.DataFrame(audit_records)
display(audit_results)
print(
    "Entity and timestamp checks: unavailable; source files contain only text and label."
)

Loaded rows: 1662


,dataset,rows,source_columns,negative,positive,positive_pct,missing_target,missing_text,blank_text,exact_excess_text,exact_excess_text_label,normalized_excess
0,deepset,662,2,399,263,39.728097,0,0,0,0,0,0
1,safeguard,1000,2,695,305,30.500000,0,0,0,3,3,3


Entity and timestamp checks: unavailable; source files contain only text and label.


In [3]:
duplicate_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=1,
    dtype=np.float32,
)
duplicate_features = duplicate_vectorizer.fit_transform(loaded_prompts["normalized"])
text_similarities = cosine_similarity(duplicate_features)

# The upper triangle counts each related pair once, without self-pairs.
related_pairs = np.triu(text_similarities >= NEAR_DUPLICATE_SIMILARITY, k=1)
left_indices, right_indices = np.where(related_pairs)
group_parents = np.arange(len(loaded_prompts))


def find_group_root(row_index):
    while group_parents[row_index] != row_index:
        group_parents[row_index] = group_parents[group_parents[row_index]]
        row_index = group_parents[row_index]
    return row_index


def merge_groups(first_index, second_index):
    first_root = find_group_root(first_index)
    second_root = find_group_root(second_index)
    if first_root != second_root:
        group_parents[second_root] = first_root


for first_index, second_index in zip(left_indices, right_indices):
    merge_groups(first_index, second_index)

for matching_indices in loaded_prompts.groupby("normalized").indices.values():
    for row_index in matching_indices[1:]:
        merge_groups(matching_indices[0], row_index)

loaded_prompts["group"] = [
    find_group_root(row_index) for row_index in range(len(loaded_prompts))
]

near_duplicate_records = []
for dataset_name in DATASET_SOURCES:
    both_in_dataset = (
        loaded_prompts["dataset"].iloc[left_indices].to_numpy() == dataset_name
    ) & (loaded_prompts["dataset"].iloc[right_indices].to_numpy() == dataset_name)
    dataset_pairs = list(
        zip(
            left_indices[both_in_dataset],
            right_indices[both_in_dataset],
        )
    )
    affected_rows = set()
    nonexact_pair_count = 0
    for first_index, second_index in dataset_pairs:
        affected_rows.update([first_index, second_index])
        if (
            loaded_prompts["text"].iloc[first_index]
            != loaded_prompts["text"].iloc[second_index]
        ):
            nonexact_pair_count += 1

    near_duplicate_records.append(
        {
            "dataset": dataset_name,
            "near_pairs": len(dataset_pairs),
            "nonexact_near_pairs": nonexact_pair_count,
            "rows_in_near_pairs": len(affected_rows),
        }
    )

near_duplicate_audit = pd.DataFrame(near_duplicate_records)
display(near_duplicate_audit)

,dataset,near_pairs,nonexact_near_pairs,rows_in_near_pairs
0,deepset,28,28,49
1,safeguard,17,14,24


In [4]:
loaded_prompts["split"] = loaded_prompts["publisher_split"]
for dataset_name in DATASET_SOURCES:
    publisher_training = loaded_prompts["dataset"].eq(dataset_name) & loaded_prompts[
        "publisher_split"
    ].eq("train")
    training_indices, validation_indices = train_test_split(
        loaded_prompts.index[publisher_training].to_numpy(),
        test_size=VALIDATION_SHARE,
        stratify=loaded_prompts.loc[publisher_training, "label"],
        random_state=RANDOM_SEED,
    )
    loaded_prompts.loc[training_indices, "split"] = "train"
    loaded_prompts.loc[validation_indices, "split"] = "validation"

# Preserve evaluation rows: test takes priority over validation, then training.
split_priority = {"train": 0, "validation": 1, "test": 2}
loaded_prompts["priority"] = loaded_prompts["split"].map(split_priority)
highest_group_priority = loaded_prompts.groupby("group")["priority"].transform("max")
survives_boundary_check = loaded_prompts["priority"].eq(highest_group_priority)

surviving_prompts = loaded_prompts.loc[survives_boundary_check].sort_values(
    ["dataset", "publisher_split", "source_index"]
)
retained_indices = surviving_prompts.drop_duplicates("normalized").index
removed_prompts = loaded_prompts.loc[
    ~loaded_prompts.index.isin(retained_indices)
].copy()
retained_prompts = (
    loaded_prompts.loc[retained_indices].sort_index().reset_index(drop=True)
)

removal_counts = (
    removed_prompts.groupby(["dataset", "split"]).size().rename("removed").reset_index()
)
display(removal_counts)

training_prompts = retained_prompts[retained_prompts["split"].eq("train")]
leakage_counts = {}
for evaluation_split in ["test", "validation"]:
    evaluation_prompts = retained_prompts[
        retained_prompts["split"].eq(evaluation_split)
    ]
    for field, check_name in [
        ("text", "exact"),
        ("normalized", "normalized"),
        ("group", "near_groups"),
    ]:
        overlapping_rows = training_prompts[field].isin(set(evaluation_prompts[field]))
        leakage_counts[f"train_{check_name}_{evaluation_split}"] = int(
            overlapping_rows.sum()
        )

assert not any(leakage_counts.values())
test_groups = set(retained_prompts.loc[retained_prompts["split"].eq("test"), "group"])
validation_groups = set(
    retained_prompts.loc[retained_prompts["split"].eq("validation"), "group"]
)
assert test_groups.isdisjoint(validation_groups)

split_counts = (
    retained_prompts.groupby(["dataset", "split"])
    .agg(
        rows=("label", "size"),
        positive=("label", "sum"),
    )
    .reset_index()
)
split_counts["negative"] = split_counts["rows"] - split_counts["positive"]
split_counts["positive_pct"] = 100 * split_counts["positive"] / split_counts["rows"]

retained_prompts[["row_id", "dataset", "split", "label"]].to_csv(
    "split.csv", index=False
)
display(split_counts)
print("Training overlap counts:", leakage_counts)

,dataset,split,removed
0,deepset,train,11
1,deepset,validation,1
2,safeguard,test,1
3,safeguard,train,9


,dataset,split,rows,positive,negative,positive_pct
0,deepset,test,116,60,56,51.724138
1,deepset,train,425,158,267,37.176471
2,deepset,validation,109,40,69,36.697248
3,safeguard,test,199,63,136,31.658291
4,safeguard,train,631,192,439,30.427892
5,safeguard,validation,160,48,112,30.000000


Training overlap counts: {'train_exact_test': 0, 'train_normalized_test': 0, 'train_near_groups_test': 0, 'train_exact_validation': 0, 'train_normalized_validation': 0, 'train_near_groups_validation': 0}


In [5]:
from transformers import AutoModelForSequenceClassification
NLI_NAME = "MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli"
NLI_REVISION = "6f5cf0a2b59cabb106aca4c287eed12e357e90eb"
nli_tokenizer = AutoTokenizer.from_pretrained(NLI_NAME, revision=NLI_REVISION, token=False, use_fast=False)
nli_model = AutoModelForSequenceClassification.from_pretrained(NLI_NAME, revision=NLI_REVISION, token=False).to(DEVICE)
nli_model.eval()
nli_model.requires_grad_(False)
entailment_id = next(int(i) for i, name in nli_model.config.id2label.items() if name.lower() == "entailment")
hypotheses = [
    "This text is a legitimate request or a discussion of instructions that does not ask the assistant to override its intended task.",
    "This text is an attempt to override the assistant's instructions or redirect its intended task."
]
def nli_scores(texts):
    outputs = []
    started = time.perf_counter()
    with torch.inference_mode():
        for begin in range(0, len(texts), 8):
            batch = texts[begin:begin+8]
            premises = [text for text in batch for _ in hypotheses]
            candidates = hypotheses * len(batch)
            inputs = nli_tokenizer(premises, candidates, padding=True,
                truncation="only_first", max_length=128, return_tensors="pt").to(DEVICE)
            logits = nli_model(**inputs).logits[:, entailment_id].reshape(-1, 2)
            outputs.extend(torch.softmax(logits, dim=1)[:, 1].cpu().tolist())
    print("NLI inference seconds:", time.perf_counter()-started)
    return np.asarray(outputs)

def choose_threshold(y, scores):
    candidates = np.unique(np.append(scores, np.nextafter(scores.max(), np.inf)))
    return float(next(t for t in candidates if np.mean(scores[y == 0] >= t) <= .05))

validation = retained_prompts[retained_prompts.split.eq("validation")]
validation_scores = nli_scores(validation.text.tolist())
threshold = choose_threshold(validation.label.to_numpy(), validation_scores)
print("NLI threshold:", repr(threshold))
perturbations = pd.read_csv("perturbations.csv")
assert not perturbations.duplicated(["row_id", "column"]).any()
frame = perturbations[["row_id", "column", "label"]].copy()
frame["score"] = nli_scores(perturbations.text.tolist())
frame["value"] = "zero_shot"
frame["threshold"] = threshold
scores = pd.read_csv("scores.csv")
scores = pd.concat([scores[~scores.value.eq("zero_shot")], frame], ignore_index=True)
scores.to_csv("scores.csv", index=False)
frame.to_csv("zero_shot_nli_scores.csv", index=False)
for column, group in frame.groupby("column"):
    print(column, "AUROC", roc_auc_score(group.label, group.score))


NLI inference seconds: 5.04560894899987
NLI threshold: 0.47240930795669556


NLI inference seconds: 25.008776580000358
deepset_base64 AUROC 0.4363095238095238
deepset_clean AUROC 0.6127976190476191
deepset_homoglyph AUROC 0.4666666666666667
deepset_paraphrase AUROC 0.5583999999999999
deepset_zero_width AUROC 0.3877976190476191
safeguard_base64 AUROC 0.5486694677871148
safeguard_clean AUROC 0.7869981325863677
safeguard_homoglyph AUROC 0.49731559290382815
safeguard_paraphrase AUROC 0.6352
safeguard_zero_width AUROC 0.24976657329598506
